# Module 05 Lab - Data Preparation

**Objective:** To learn and apply the most common data preparation techniques. Raw data is rarely ready for a machine learning model. This process, also called preprocessing, is one of the most critical steps in the entire ML workflow.

**In this lab, you will write more of the code.** Read the explanations and then complete the tasks in the code cells.

## Part 1: Setup and Initial Look

We will continue using the Titanic dataset because it has the exact problems we need to solve: missing values and non-numeric data.

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Let's look at the missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

## Part 2: Handling Missing Values (Imputation)

**Concept:** Most machine learning models cannot handle missing values (`NaN`). We must deal with them. Dropping the rows is an option, but you lose data. A better way is **imputation**, which means filling in the missing values with a calculated guess.

Common imputation strategies:
*   **Mean:** Fill with the average value. Good for normally distributed data.
*   **Median:** Fill with the middle value. Better for skewed data or data with outliers (like `Fare`).
*   **Mode:** Fill with the most frequent value. Used for categorical data.

### Task 1: Impute the 'Age' Column

The 'Age' column is missing many values. Since age can be skewed (e.g., by a few very old passengers), using the **median** is a robust choice.

**Your Task:** Calculate the median of the 'Age' column and use the `.fillna()` method to replace the missing values.

In [ ]:
# --- ENTER YOUR CODE HERE ---
# 1. Calculate the median of the 'Age' column
#    (median is robust to the skew caused by a few very old passengers)
median_age = df['Age'].median()
print(f"Median age used for imputation: {median_age}")

# 2. Fill the missing values in 'Age' with the median
#    Note: we assign the result back instead of using inplace=True on a single column,
#    because newer versions of pandas warn that chained inplace operations may not
#    modify the original DataFrame.
df['Age'] = df['Age'].fillna(median_age)

# Extra: 'Embarked' is categorical and has a couple of missing values,
# so we fill it with the MODE (most frequent port) before encoding it in Task 2.
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Mode of 'Embarked' used for imputation: {mode_embarked}")

# 3. Verify that there are no more missing values in 'Age'
print("Missing values in 'Age' after imputation:")
print(df['Age'].isnull().sum())

print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())

## Part 3: Encoding Categorical Features

**Concept:** Machine learning models are mathematical, so they need numbers, not text. We need to convert categorical columns (like 'Sex' and 'Embarked') into a numerical format. The most common method is **One-Hot Encoding**.

One-Hot Encoding takes a column with `N` categories and turns it into `N` new columns, each with a `1` or `0`. For example, the 'Sex' column (`male`, `female`) becomes two new columns: `Sex_male` and `Sex_female`.

Pandas has a convenient function called `pd.get_dummies()` that does this for us.

### Task 2: One-Hot Encode Categorical Columns

**Your Task:** Use `pd.get_dummies()` to encode the 'Sex' and 'Embarked' columns. Make sure to drop the original columns after encoding.

In [ ]:
# --- ENTER YOUR CODE HERE ---
# 1. Use get_dummies to create new columns for 'Sex' and 'Embarked'
#    drop_first=True drops one dummy per feature to avoid multicollinearity
#    (e.g., Sex_male alone tells us whether a passenger is female).
#    dtype=int shows the new columns as 1/0 instead of True/False.
encoded_df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)

# 2. Display the first few rows of the new DataFrame to see the new columns
print("New columns created:", [c for c in encoded_df.columns if c.startswith(('Sex_', 'Embarked_'))])
print(encoded_df.head())

## Part 4: Feature Scaling

**Concept:** Many models are sensitive to the scale of the features. For example, `Age` (from 0-80) and `Fare` (from 0-512) are on very different scales. This can cause the model to incorrectly believe that `Fare` is a more important feature simply because its values are larger.

**Feature Scaling** solves this by putting all features on a similar scale. A common method is **Standardization** (`StandardScaler` in scikit-learn), which rescales the data to have a mean of 0 and a standard deviation of 1.

**Important:** You only scale your numerical features, not your target variable or your newly encoded categorical columns.

### Task 3: Scale the 'Age' and 'Fare' Columns

**Your Task:** Use `StandardScaler` from `sklearn.preprocessing` to scale the 'Age' and 'Fare' columns.

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- ENTER YOUR CODE HERE ---
# 1. Create an instance of the StandardScaler
scaler = StandardScaler()

# 2. Select the columns to scale (numerical features only, not the target 'Survived'
#    and not the 0/1 encoded columns)
columns_to_scale = ['Age', 'Fare']

# 3. Fit the scaler to the data and transform it
#    fit() learns each column's mean and standard deviation; transform() applies
#    z = (x - mean) / std so each column ends up with mean 0 and std 1.
encoded_df[columns_to_scale] = scaler.fit_transform(encoded_df[columns_to_scale])

# 4. Display the first few rows to see the scaled data
print(encoded_df.head())

# Wide tables get truncated with '...', so also show the scaled columns on their own
print("\nScaled columns (first 5 rows):")
print(encoded_df[columns_to_scale].head())

# Verify the result: means should be ~0 and standard deviations ~1
print("\nMean after scaling:")
print(encoded_df[columns_to_scale].mean().round(4))
print("\nStandard deviation after scaling:")
print(encoded_df[columns_to_scale].std(ddof=0).round(4))

## 📝 Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **Why is it often better to impute missing values with the median instead of the mean?**
2.  **Explain in your own words what One-Hot Encoding does and why it is necessary.**
3.  **Would you need to apply Feature Scaling to a Decision Tree model?** Why or why not? (Hint: Think about how a Decision Tree makes its splits).

---

**Answer 1 – Median vs. mean:**
The mean is pulled toward extreme values, while the median is just the middle value once the data is sorted, so a few outliers barely move it. In the Titanic data, a handful of elderly passengers make `Age` slightly right-skewed, and `Fare` is very skewed (most people paid under 30, but some paid over 500). If we filled missing values with the mean, those extremes would decide the value we fill in, and it might not look like a typical passenger. The median gives a more realistic "typical" value and keeps the distribution's center intact. The mean is only a safe choice when the data is roughly symmetric and has no strong outliers.

**Answer 2 – One-Hot Encoding:**
One-Hot Encoding turns a text category into a set of yes/no (1/0) columns, one for each possible category. For example, `Embarked` (C, Q, S) becomes columns like `Embarked_Q` and `Embarked_S`, and each passenger gets a 1 in the column for their port. It is necessary because ML models do math on numbers and cannot read strings like "male" or "S". We also cannot just number the categories (C=0, Q=1, S=2), because that would invent a false order and distance, as if S were "twice" Q. One-hot columns let the model treat each category as separate and equal. With `drop_first=True` we drop one redundant column, since a passenger with 0 in every remaining dummy must belong to the dropped category. This avoids multicollinearity.

**Answer 3 – Feature scaling for a Decision Tree:**
No, a Decision Tree does not need feature scaling. A tree makes splits by asking threshold questions about one feature at a time, such as "Is Age ≤ 30?" or "Is Fare ≤ 50?". It picks the threshold that best separates the classes (for example, by reducing Gini impurity). Scaling a feature changes its units but not the order of its values, so the same passengers end up on each side of the split and the tree's results stay the same. Features are never combined in a distance or weighted sum, so a large-valued feature like `Fare` cannot dominate just because its numbers are bigger. Scaling matters for distance-based or gradient-based models such as KNN, SVM, logistic or linear regression, and neural networks. It is harmless, but unnecessary, for trees and tree ensembles like Random Forest.